# LangSmith essentials: see, debug, and evaluate an LLM application
**Beginner lesson · 35 minutes (up to 40 with discussion) · Python 3.11+.** Install packages and prepare accounts before class.

## What is LangSmith?
LangSmith is a platform for **observing, debugging, and evaluating LLM applications**. It records a request as a trace containing the steps that ran, their inputs and outputs, timing, and errors. It also organizes labeled datasets and experiments so you can compare application changes.

**Problem it solves:** a chatbot's final answer does not tell you which prompt it saw, which step failed, or whether a new prompt helped on other examples. LangSmith connects those details to the request and lets you test changes systematically.

LangChain builds application components; LangSmith helps inspect and evaluate them. You can use LangSmith with plain Python and provider SDKs, as we do here. It does not supply an LLM or automatically make answers correct.

## What this notebook covers
| Time | Essential topic | What you will see |
|---|---|---|
| 4 min | Vocabulary and purpose | Traces, runs, datasets, evaluators, experiments |
| 6 min | Setup and provider selection | Gemini, OpenAI, or hosted Hugging Face |
| 10 min | Trace and debug | Nested prompt/model steps and a deliberate error |
| 10 min | Evaluate and compare | Two prompts, four labeled tickets, exact-match scores |
| 5 min | Interpret and practice | Inspect failures and explain the limits of a tiny evaluation |

Our one task is to route synthetic support tickets to **billing**, **security**, or **technical**. We keep the application small so the focus stays on LangSmith. Advanced agents, RAG, deployment, and LLM-as-judge evaluators are outside this lesson.

| Term | Meaning in this lesson |
|---|---|
| Trace | The whole request, including nested steps |
| Run / span | One executed function inside that trace |
| Dataset / example | A collection / one input with its reference answer |
| Evaluator / feedback | A scoring function / the score it produces |
| Experiment | One application version tested on the dataset |

## 1. Install (before class)
Uncomment the first line once and restart the kernel if needed. The core lesson needs `langsmith`, `pandas`, and `python-dotenv`. Install only the provider SDK you plan to use. The notebook contains all sample data; no file downloads or vector database are required.

In [1]:
# %pip install -U langsmith pandas python-dotenv
# %pip install -U google-genai      # Gemini
# %pip install -U openai            # OpenAI
# %pip install -U huggingface_hub   # Hosted Hugging Face inference
print('Install into this notebook kernel, then continue.')

Install into this notebook kernel, then continue.


## 2. Choose local demonstration or hosted LangSmith
There are **two independent switches**:
- `USE_DEMO_MODEL=True` uses a tiny deterministic keyword rule, not an LLM. Set it to `False` to call the provider selected by `PROVIDER`.
- `ENABLE_LANGSMITH=True` sends traces, examples, and evaluation results to your LangSmith account. The default `False` is a local walkthrough; it does **not** provide a hosted trace dashboard or pretend to be an offline LangSmith server.

You can enable LangSmith while keeping the demo model to teach the real dashboard without paying for model calls. To teach the full LLM workflow, disable the demo and enable LangSmith.

Copy `.env.example` to `.env` in this directory and fill in keys. This lesson loads only that local file. Keep `.env` out of source control. Model IDs are explicit because availability varies by account; choose a text-generation/chat model supported by your provider.

| Provider | Key | Model variable |
|---|---|---|
| Gemini (default choice) | GEMINI_API_KEY | GEMINI_MODEL |
| OpenAI | OPENAI_API_KEY | OPENAI_MODEL |
| Hugging Face | HF_TOKEN | HF_MODEL |

Hugging Face here uses hosted inference, not a local Transformers model. Provider calls may be billable. One normal run of the notebook makes **9 model calls** with a real provider: one sample and eight evaluation calls. The deliberate validation failure makes none. Re-running cells makes additional calls.

### LangSmith account setup
Create a LangSmith API key and set `LANGSMITH_API_KEY`. Use your region's API URL in `LANGSMITH_ENDPOINT`; the default is US.
- **Project-based workspace:** leave `LANGSMITH_AGENT_ID` empty; the lesson uses the tracing project `langsmith-student-lab`.
- **Agent-based workspace:** create an Observability agent in the UI, then set its `LANGSMITH_AGENT_ID` and `LANGSMITH_AGENT_ENVIRONMENT=local`. Use an up-to-date SDK supporting `agent_address`; do not combine agent addressing with legacy project environment settings.
- If your key covers multiple workspaces, set `LANGSMITH_WORKSPACE_ID` as well.

**Expected below:** mode, provider, and SDK version only. Missing keys fail clearly when their feature is enabled; secrets are not printed.

In [2]:
import os, time
from pathlib import Path
from uuid import uuid4
from importlib.metadata import version
from contextlib import contextmanager
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from langsmith import Client, traceable, tracing_context

load_dotenv(Path.cwd() / '.env', override=False)
USE_DEMO_MODEL = True
ENABLE_LANGSMITH = False
PROVIDER = 'gemini'  # 'gemini', 'openai', or 'huggingface'
PROJECT = 'langsmith-student-lab'


def required(name):
    value = os.getenv(name)
    if not value:
        raise ValueError(f'Set {name} in your environment or local .env file.')
    return value


def selected_model():
    if USE_DEMO_MODEL:
        return 'keyword-demo-v1'
    names = {'gemini': 'GEMINI_MODEL', 'openai': 'OPENAI_MODEL', 'huggingface': 'HF_MODEL'}
    if PROVIDER not in names:
        raise ValueError('Choose gemini, openai, or huggingface.')
    return required(names[PROVIDER])


ls_client = None
trace_destination = {'project_name': PROJECT}
if ENABLE_LANGSMITH:
    ls_client = Client(api_key=required('LANGSMITH_API_KEY'),
                       api_url=os.getenv('LANGSMITH_ENDPOINT') or 'https://api.smith.langchain.com',
                       workspace_id=os.getenv('LANGSMITH_WORKSPACE_ID') or None)
    if os.getenv('LANGSMITH_AGENT_ID'):
        import langsmith
        if not hasattr(langsmith, 'agent_address'):
            raise RuntimeError('Agent-based workspace: upgrade langsmith and restart the kernel.')
        legacy = ['LANGSMITH_PROJECT', 'LANGCHAIN_PROJECT', 'LANGCHAIN_SESSION']
        if any(os.getenv(name) for name in legacy):
            raise ValueError('Remove legacy project variables when using agent addressing.')
        trace_destination = {'address': langsmith.agent_address(
            os.environ['LANGSMITH_AGENT_ID'], os.getenv('LANGSMITH_AGENT_ENVIRONMENT', 'local'))}


@contextmanager
def lesson_tracing():
    # Explicitly overrides ambient tracing settings, including when disabled.
    with tracing_context(enabled=ENABLE_LANGSMITH, client=ls_client,
                         metadata={'provider': 'demo' if USE_DEMO_MODEL else PROVIDER,
                                   'model': selected_model()}, **trace_destination):
        yield

print({'model_mode': 'local demo' if USE_DEMO_MODEL else 'real provider',
       'provider': PROVIDER, 'model': selected_model(),
       'hosted_langsmith': ENABLE_LANGSMITH, 'langsmith_version': version('langsmith')})

{'model_mode': 'local demo', 'provider': 'gemini', 'model': 'keyword-demo-v1', 'hosted_langsmith': False, 'langsmith_version': '0.12.1'}


## 3. A swappable model adapter
All providers return one string to the same classifier. You only change `PROVIDER` and its environment variables; evaluation code stays the same. Client objects are reused. Keys are read inside the adapter, not passed as traced inputs.

`@traceable` marks a function as a span. `run_type='llm'` identifies the model step; child spans are attached automatically when a traced function calls another one. This manual adapter focuses on prompts, outputs, duration, and errors. **It does not normalize token usage or calculate provider costs**; automatic provider wrappers/integrations can add richer model telemetry.

**Expected:** definitions only; no requests yet. The output limit is a budget, not a promise of an exact length; some reasoning models need a larger budget.

In [ ]:
provider_clients = {}

@traceable(name='call_model', run_type='llm')
def call_model(prompt: str) -> str:
    if USE_DEMO_MODEL:
        # Deliberately naive: ignores negation and prompt instructions.
        ticket = prompt.rsplit('Ticket:', 1)[-1].lower()
        if any(word in ticket for word in ['invoice', 'charge', 'refund']):
            return 'billing'
        if any(word in ticket for word in ['password', 'login', 'recovery']):
            return 'security'
        return 'technical'

    model = selected_model()
    if PROVIDER == 'gemini':
        from google import genai
        from google.genai import types
        if PROVIDER not in provider_clients:
            provider_clients[PROVIDER] = genai.Client(
                api_key=required('GEMINI_API_KEY'),
                http_options=types.HttpOptions(timeout=60000))
        response = provider_clients[PROVIDER].models.generate_content(
            model=model, contents=prompt,
            config=types.GenerateContentConfig(max_output_tokens=256))
        return (response.text or '').strip()
    
    if PROVIDER == 'openai':
        from openai import OpenAI
        if PROVIDER not in provider_clients:
            provider_clients[PROVIDER] = OpenAI(
                api_key=required('OPENAI_API_KEY'), timeout=60, max_retries=0)
        response = provider_clients[PROVIDER].responses.create(
            model=model, input=prompt, max_output_tokens=256)
        return response.output_text.strip()
    
    if PROVIDER == 'huggingface':
        from huggingface_hub import InferenceClient
        if PROVIDER not in provider_clients:
            provider_clients[PROVIDER] = InferenceClient(token=required('HF_TOKEN'), timeout=60)
        response = provider_clients[PROVIDER].chat_completion(
            model=model, messages=[{'role': 'user', 'content': prompt}], max_tokens=256)
        return (response.choices[0].message.content or '').strip()
    raise ValueError('Unknown provider.')

## 4. Trace one complete request
We separate prompt construction and model invocation. A hosted trace should have this shape:

```text
classify_ticket
├── build_prompt
└── call_model
```

The first prompt asks for a category; the second defines routing rules and demands a label-only answer. This is a change worth measuring, not assuming is better.

**Expected:** `billing` in demo mode. With LangSmith enabled, open the tracing project (or agent's local environment), locate `classify_ticket`, expand it, inspect the exact prompt and model output, and compare parent/child durations. The SDK sends traces in the background; `flush()` waits for pending uploads. Network or permission errors must be resolved before relying on the dashboard.

In [4]:
PROMPTS = {
    'v1': 'Classify this support ticket as billing, security, or technical.\nTicket: {ticket}',
    'v2': ('Route by the actual requested help, not incidental keywords. '
           'billing: payments or subscriptions; security: account access; '
           'technical: software or upload problems. '
           'Output exactly one lowercase label: billing, security, or technical.\n'
           'Ticket: {ticket}'),
}

@traceable(name='build_prompt', run_type='prompt')
def build_prompt(ticket: str, prompt_version: str) -> str:
    return PROMPTS[prompt_version].format(ticket=ticket)

@traceable(name='classify_ticket')
def classify_ticket(inputs: dict, prompt_version: str = 'v1') -> dict:
    ticket = inputs.get('ticket', '')
    if not isinstance(ticket, str) or not ticket.strip():
        raise ValueError('A non-empty ticket is required.')
    prompt = build_prompt(ticket, prompt_version)
    raw = call_model(prompt)
    # Keep raw text for debugging; only trim whitespace/case for scoring.
    return {'label': raw.strip().lower(), 'raw': raw}

with lesson_tracing():
    sample = classify_ticket({'ticket': 'Please send me the invoice.'}, 'v1')
print(sample)
if ls_client:
    ls_client.flush()
    print('Open LangSmith and inspect the classify_ticket trace.')

{'label': 'billing', 'raw': 'billing'}


## 5. Debug a deliberate failure
An empty support ticket is an application error, not an LLM error. Raising the exception **inside** the traced function records the failure when tracing is enabled. Catch it outside so the notebook can continue.

**Expected:** `ValueError: A non-empty ticket is required.` In LangSmith, find the failed trace and notice that neither prompt-building nor a model call ran. The trace explains *where* the request stopped; the printed message alone does not show that hierarchy.

In [5]:
with lesson_tracing():
    try:
        classify_ticket({'ticket': ''})
    except ValueError as exc:
        print(f'{type(exc).__name__}: {exc}')
if ls_client:
    ls_client.flush()

ValueError: A non-empty ticket is required.


## 6. Define a tiny dataset and a code evaluator
A dataset pairs inputs with reference outputs. An evaluator turns each prediction into **feedback**: here `exact_match` is 1 for a matching label and 0 otherwise. No judge LLM or extra model request is needed to score a response.

The last example mentions an invoice but asks about a freezing app. It exposes the demo rule's keyword mistake. These four rows are teaching examples, not a credible benchmark.

**Expected:** four input/reference pairs; the scorer gives 1 for a correct answer and 0 for a wrong answer. Extra explanation or punctuation in the output fails this strict format requirement.

In [6]:
EXAMPLES = [
    {'inputs': {'ticket': 'Please send me the invoice.'}, 'outputs': {'label': 'billing'}},
    {'inputs': {'ticket': 'I lost my password and need account access.'}, 'outputs': {'label': 'security'}},
    {'inputs': {'ticket': 'My PDF upload keeps failing.'}, 'outputs': {'label': 'technical'}},
    {'inputs': {'ticket': 'My invoice is correct. The app freezes when I open a PDF.'},
     'outputs': {'label': 'technical'}},
]

def exact_match(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return {'key': 'exact_match',
            'score': int(outputs.get('label') == reference_outputs['label'])}

for example in EXAMPLES:
    print(example)
print('Correct:', exact_match({}, {'label': 'billing'}, {'label': 'billing'}))
print('Wrong:  ', exact_match({}, {'label': 'technical'}, {'label': 'billing'}))

{'inputs': {'ticket': 'Please send me the invoice.'}, 'outputs': {'label': 'billing'}}
{'inputs': {'ticket': 'I lost my password and need account access.'}, 'outputs': {'label': 'security'}}
{'inputs': {'ticket': 'My PDF upload keeps failing.'}, 'outputs': {'label': 'technical'}}
{'inputs': {'ticket': 'My invoice is correct. The app freezes when I open a PDF.'}, 'outputs': {'label': 'technical'}}
Correct: {'key': 'exact_match', 'score': 1}
Wrong:   {'key': 'exact_match', 'score': 0}


## 7. Compare two experiments on the same dataset
In hosted mode, `Client.create_dataset` / `create_examples` stores the four examples, and `Client.evaluate` calls the target once per example and attaches evaluator feedback. The two experiment prefixes identify the prompt versions. The SDK manages experiment runs and reference-example links; we do not build that bookkeeping ourselves.

In local mode, a plain loop shows the same task and scorer, without claiming it created LangSmith experiments. **Only one branch executes**, so the evaluation makes eight model calls, not sixteen. This deliberate failure example is excluded from accuracy scoring. A real provider error stops the local loop; hosted evaluation records target errors—inspect errored rows as well as aggregate scores.

**Expected in demo mode:** both prompt versions score **3/4 = 0.75**. The keyword rule ignores instructions, so prompt v2 cannot fix it. Real model results may differ; v2 is not guaranteed to improve the score.

Hosted mode uploads the synthetic dataset and creates two experiments. Each rerun uses a unique dataset name so classroom runs do not silently accumulate duplicate examples. In an ongoing project, reuse and version a fixed benchmark instead.

In [7]:
def target_v1(inputs: dict) -> dict:
    return classify_ticket(inputs, 'v1')

def target_v2(inputs: dict) -> dict:
    return classify_ticket(inputs, 'v2')

targets = {'v1': target_v1, 'v2': target_v2}
if ENABLE_LANGSMITH:
    dataset = ls_client.create_dataset(
        dataset_name=f'student-ticket-routing-{uuid4().hex[:8]}',
        description='Four synthetic tickets for a beginner LangSmith lesson.')
    ls_client.create_examples(dataset_id=dataset.id, examples=EXAMPLES)
    hosted_results = {}
    with lesson_tracing():
        for prompt_version, target in targets.items():
            result = ls_client.evaluate(
                target, data=dataset.name, evaluators=[exact_match],
                experiment_prefix=f'{PROVIDER if not USE_DEMO_MODEL else "demo"}-{prompt_version}',
                metadata={'provider': 'demo' if USE_DEMO_MODEL else PROVIDER,
                          'model': selected_model(), 'prompt_version': prompt_version,
                          'prompt_template': PROMPTS[prompt_version]},
                max_concurrency=1, num_repetitions=1, blocking=True)
            hosted_results[prompt_version] = result
            display(result.to_pandas())
    ls_client.flush()
    print('Dataset:', dataset.name)
    print('In LangSmith, open this dataset and compare its two experiments.')
else:
    rows = []
    with lesson_tracing():
        for prompt_version, target in targets.items():
            for example in EXAMPLES:
                started = time.perf_counter()
                output = target(example['inputs'])
                score = exact_match(example['inputs'], output, example['outputs'])['score']
                rows.append({'prompt': prompt_version,
                             'ticket': example['inputs']['ticket'],
                             'expected': example['outputs']['label'],
                             'predicted': output['label'], 'exact_match': score,
                             'duration_ms': (time.perf_counter() - started) * 1000})
    local_results = pd.DataFrame(rows)
    display(local_results)
    display(local_results.groupby('prompt', as_index=False)['exact_match'].mean())
    print('Local demonstration only: enable LangSmith to create hosted experiments.')

  prompt  ... duration_ms
0     v1  ...      0.0854
1     v1  ...      0.0637
2     v1  ...      0.0445
3     v1  ...      0.0433
4     v2  ...      0.0399
5     v2  ...      0.0441
6     v2  ...      0.0427
7     v2  ...      0.0398

[8 rows x 6 columns]
  prompt  exact_match
0     v1         0.75
1     v2         0.75
Local demonstration only: enable LangSmith to create hosted experiments.


prompt,ticket,expected,predicted,exact_match,duration_ms
v1,Please send me the invoice.,billing,billing,1,0.0854
v1,I lost my password and need account access.,security,security,1,0.0637
v1,My PDF upload keeps failing.,technical,technical,1,0.0445
v1,My invoice is correct. The app freezes when I open a PDF.,technical,billing,0,0.0433
v2,Please send me the invoice.,billing,billing,1,0.0399
v2,I lost my password and need account access.,security,security,1,0.0441
v2,My PDF upload keeps failing.,technical,technical,1,0.0427
v2,My invoice is correct. The app freezes when I open a PDF.,technical,billing,0,0.0398


prompt,exact_match
v1,0.75
v2,0.75


## 8. Read the results and try one change (5 minutes)
With hosted LangSmith enabled:
1. Open the dataset and select the two experiments for comparison.
2. Compare `exact_match`, latency, and any errors. Open an incorrect row and inspect the reference answer, actual output, and nested prompt/model steps.
3. Check the last ticket: did the model identify the actual request or just match the word “invoice”?
4. Switch `PROVIDER`, provide its model and key, restart the kernel, and rerun. The adapter, tracing, dataset format, and evaluator stay the same. New hosted runs create a separate classroom dataset; compare provider results manually, or reuse the original dataset for a longer follow-up exercise.

**Mini exercise:** add a ticket about canceling a subscription. Write the reference label *before* seeing the model output, then rerun the evaluation. Explain one mistake using its trace.

### What to remember
- **Tracing asks “what happened in this request?” Evaluation asks “how well did this version perform across examples?”** They answer different questions.
- A trace can reveal a bad prompt or an API failure; it does not itself judge correctness.
- A strict label scorer is appropriate here, not for grading open-ended essays. More complex tasks need better rubrics and representative data.
- Four examples cannot establish model superiority. Use a larger development set, separate held-out test data, and repeat measurements before choosing a model.
- Local demo latency is Python execution time, not a model performance measurement. Manually traced model calls in this notebook do not claim accurate token or cost telemetry.
- Hosted tracing uploads function inputs and outputs, including prompts and ticket text. Only use data approved for sharing; do not put API keys in traced inputs.

### Troubleshooting
| Symptom | Check |
|---|---|
| No hosted traces | Enable LangSmith, rerun setup and request cells, check destination and credentials, flush pending uploads |
| 401/403 | LangSmith region/endpoint, workspace access, or the selected provider key |
| Agent-address error | Upgrade LangSmith, restart kernel, configure one destination style |
| Model not found | Use an available chat/text model; HF must have hosted inference support |
| 429 | Provider quota/rate limit; avoid repeatedly running the batch |
| Blank or verbose label | Inspect the model span; check output budget and instruction adherence |
| Prompt v2 does not help demo | Expected: the keyword rule is not an LLM and ignores instructions |
| SDK import error | Install into this kernel and restart it |

### Official references
Checked 2026-11-06; account UI and SDK features can evolve.
- [LangSmith tracing quickstart](https://docs.langchain.com/langsmith/observability-quickstart)
- [Custom tracing with traceable](https://docs.langchain.com/langsmith/annotate-code)
- [Tracing context and destination configuration](https://docs.langchain.com/langsmith/trace-without-env-vars)
- [Run evaluations with the SDK](https://docs.langchain.com/langsmith/evaluate-llm-application)
- [Evaluation concepts and types](https://docs.langchain.com/langsmith/evaluation-types)
- [Gemini generation API](https://ai.google.dev/api/generate-content)
- [OpenAI text generation](https://platform.openai.com/docs/guides/text)
- [Hugging Face hosted inference client](https://huggingface.co/docs/huggingface_hub/en/package_reference/inference_client)